# ML-05 — Feature Vector and Leakage/Privacy Check

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Build the feature vector

For my **Refresh / Content Opportunity Scoring** lane, the feature vector is one row per pseudonymized content page. The target proxy is `is_declining_proxy = trend_direction == "down"`.

This code builds a starter feature vector from observable page-level signals: keyword context, content depth, search visibility, traffic, engagement, freshness, and interpretable tiers. It also adds a few careful helper features, such as log-transformed traffic totals and missingness flags.

Important: this is a starter proxy workflow. Because the proxy comes from current trend fields, I exclude direct trend fields from the final safe feature vector. For a future capstone model, I should rebuild the feature and target windows so all features are known before the future outcome window.


In [1]:
import numpy as np
import pandas as pd
from pathlib import Path

candidates = [
    Path('data/raw/content_refresh_anonymized.csv'),
    Path('../../data/raw/content_refresh_anonymized.csv'),
]
data_path = next(path for path in candidates if path.exists())
raw_df = pd.read_csv(data_path)

df = raw_df.copy()
df['is_declining_proxy'] = df['trend_direction'].eq('down').astype(int)

# Engineered features: log traffic totals for heavy-tailed counts and flags for meaningful missing/no-data states.
for col in [
    'impressions_90d', 'clicks_90d', 'pageviews_90d', 'sessions_90d',
    'users_90d', 'engaged_sessions_90d', 'ai_sessions_90d', 'scroll_events_90d',
]:
    df[f'log_{col}'] = np.log1p(df[col].fillna(0))

df['has_keyword_data'] = df['search_volume'].notna().astype(int)
df['has_word_count'] = df['word_count'].notna().astype(int)
df['has_clicks'] = (df['clicks_90d'] > 0).astype(int)
df['has_sessions'] = (df['sessions_90d'] > 0).astype(int)
df['has_ai_sessions'] = (df['ai_sessions_90d'] > 0).astype(int)
df['avg_position_no_data'] = (df['avg_position'] == 0).astype(int)
df['avg_position_clean'] = df['avg_position'].replace(0, np.nan)

numeric_features = [
    'search_volume', 'competition', 'cpc', 'word_count', 'char_count',
    'log_impressions_90d', 'log_clicks_90d', 'log_pageviews_90d',
    'log_sessions_90d', 'log_users_90d', 'log_engaged_sessions_90d',
    'log_ai_sessions_90d', 'log_scroll_events_90d',
    'days_with_impressions', 'days_with_sessions', 'content_age_days',
    'age_tier_order', 'days_since_last_update', 'ctr', 'avg_position_clean',
    'engagement_rate', 'scroll_rate', 'ai_traffic_pct',
    'has_keyword_data', 'has_word_count', 'has_clicks', 'has_sessions',
    'has_ai_sessions', 'avg_position_no_data',
]

categorical_features = [
    'competition_level', 'content_type', 'main_intent', 'age_tier',
    'freshness_tier', 'word_count_tier', 'char_count_tier',
    'impression_tier', 'position_tier',
]

safe_feature_columns = numeric_features + categorical_features
feature_vector = df[['content_id', 'client_id', 'is_declining_proxy'] + safe_feature_columns].copy()

feature_summary = pd.DataFrame({
    'item': ['rows', 'unit_of_analysis', 'safe_numeric_features', 'safe_categorical_features', 'target_proxy_positive_rows', 'target_proxy_positive_share'],
    'value': [
        len(feature_vector),
        'one row = one pseudonymized content page',
        len(numeric_features),
        len(categorical_features),
        int(feature_vector['is_declining_proxy'].sum()),
        round(feature_vector['is_declining_proxy'].mean(), 3),
    ],
})

feature_summary


,item,value
0,rows,30000
1,unit_of_analysis,one row = one pseudonymized content page
2,safe_numeric_features,29
3,safe_categorical_features,9
4,target_proxy_positive_rows,16262
5,target_proxy_positive_share,0.542


## 2. Feature notes (meaning, missing, categorical, available-when?)

The safe feature vector uses only fields I would be willing to explain to a reviewer. Numeric missing values are not blindly treated as truth; I add explicit missing/no-data flags where the missingness itself matters, then the model pipeline can impute remaining numeric blanks. Categorical blanks are filled as `unknown` in the model pipeline.

All rate columns are percentage points. For example, `ctr = 0.76` means 0.76%, not 76%. `avg_position = 0` is treated as no position data, not as the best possible rank.


In [2]:
feature_notes = pd.DataFrame([
    ('search_volume / competition / cpc', 'keyword context', 'numeric', 'median impute after checking missingness; has_keyword_data flag protects meaning', 'yes, when keyword context exists'),
    ('word_count / char_count', 'content depth', 'numeric', 'median impute after checking missingness; has_word_count flag protects meaning', 'yes, content property'),
    ('log_*_90d totals', 'search, traffic, engagement, and AI-referral volume with log1p scaling', 'numeric', 'source totals fill missing as 0 before log because counts mean none/absent in this slice', 'yes for current review scoring'),
    ('days_with_impressions / days_with_sessions', 'how consistently the page had search visibility or sessions', 'numeric', 'median impute if needed', 'yes for current review scoring'),
    ('content_age_days / days_since_last_update', 'age and freshness signals', 'numeric', 'median impute if needed', 'yes, known at scoring time'),
    ('ctr / engagement_rate / scroll_rate / ai_traffic_pct', 'derived percentage-rate signals', 'numeric', 'median impute; remember these are percentage points', 'yes for current review scoring'),
    ('avg_position_clean / avg_position_no_data', 'search ranking position with zero treated as no data', 'numeric', 'replace 0 with missing and keep no-data flag', 'yes for current review scoring'),
    ('has_keyword_data / has_word_count / has_clicks / has_sessions / has_ai_sessions', 'missingness and presence flags', 'numeric binary', 'already 0/1', 'yes, known at scoring time'),
    ('content_type / main_intent / tiers', 'categorical page context and interpretable buckets', 'categorical', 'fill blank values as unknown in model pipeline', 'yes, known at scoring time'),
], columns=['feature_group', 'meaning', 'kind', 'missing_or_special_handling', 'available_when'])

feature_notes


,feature_group,meaning,kind,missing_or_special_handling,available_when
0,search_volume / competition / cpc,keyword context,numeric,median impute after checking missingness; has_...,"yes, when keyword context exists"
1,word_count / char_count,content depth,numeric,median impute after checking missingness; has_...,"yes, content property"
2,log_*_90d totals,"search, traffic, engagement, and AI-referral v...",numeric,source totals fill missing as 0 before log bec...,yes for current review scoring
3,days_with_impressions / days_with_sessions,how consistently the page had search visibilit...,numeric,median impute if needed,yes for current review scoring
4,content_age_days / days_since_last_update,age and freshness signals,numeric,median impute if needed,"yes, known at scoring time"
5,ctr / engagement_rate / scroll_rate / ai_traff...,derived percentage-rate signals,numeric,median impute; remember these are percentage p...,yes for current review scoring
6,avg_position_clean / avg_position_no_data,search ranking position with zero treated as n...,numeric,replace 0 with missing and keep no-data flag,yes for current review scoring
7,has_keyword_data / has_word_count / has_clicks...,missingness and presence flags,numeric binary,already 0/1,"yes, known at scoring time"
8,content_type / main_intent / tiers,categorical page context and interpretable buc...,categorical,fill blank values as unknown in model pipeline,"yes, known at scoring time"


## 3. The leakage hunt

Here I attack my own feature set. The target proxy is created from `trend_direction`, which is created from trend movement. Therefore `trend_direction`, `trend_pct`, and the 30-day comparison fields are suspect.

The test below trains two simple models with a client-grouped split:

- **Safe feature model:** uses the approved feature vector.
- **Leaky feature model:** adds direct trend fields, including `trend_direction`, which gives the answer away.

If the leaky model jumps toward perfect performance, that is not a win. It is evidence that the leakage test is working and those fields must stay out of the feature vector.


In [3]:
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.metrics import average_precision_score, roc_auc_score
from sklearn.model_selection import GroupShuffleSplit
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.tree import DecisionTreeClassifier

def precision_at_k(y_true, scores, k=50):
    ranked = pd.DataFrame({'y': y_true, 'score': scores}).sort_values('score', ascending=False).head(k)
    return float(ranked['y'].mean())

def make_pipeline(num_cols, cat_cols):
    preprocessor = ColumnTransformer([
        ('num', SimpleImputer(strategy='median'), num_cols),
        ('cat', Pipeline([
            ('imputer', SimpleImputer(strategy='constant', fill_value='unknown')),
            ('onehot', OneHotEncoder(handle_unknown='ignore')),
        ]), cat_cols),
    ])
    return Pipeline([
        ('prep', preprocessor),
        ('model', DecisionTreeClassifier(max_depth=5, min_samples_leaf=50, random_state=42)),
    ])

y = feature_vector['is_declining_proxy']
groups = feature_vector['client_id']
splitter = GroupShuffleSplit(n_splits=1, test_size=0.25, random_state=42)
train_idx, test_idx = next(splitter.split(feature_vector, y, groups=groups))

X_safe = feature_vector[safe_feature_columns]
safe_model = make_pipeline(numeric_features, categorical_features)
safe_model.fit(X_safe.iloc[train_idx], y.iloc[train_idx])
safe_scores = safe_model.predict_proba(X_safe.iloc[test_idx])[:, 1]

leaky_numeric_features = numeric_features + [
    'trend_pct', 'impressions_last_30d', 'clicks_last_30d', 'sessions_last_30d',
    'impressions_prev_30d', 'clicks_prev_30d', 'sessions_prev_30d',
]
leaky_categorical_features = categorical_features + ['trend_direction']
leaky_feature_columns = leaky_numeric_features + leaky_categorical_features
X_leaky = df[leaky_feature_columns]
leaky_model = make_pipeline(leaky_numeric_features, leaky_categorical_features)
leaky_model.fit(X_leaky.iloc[train_idx], y.iloc[train_idx])
leaky_scores = leaky_model.predict_proba(X_leaky.iloc[test_idx])[:, 1]

y_test = y.iloc[test_idx]
leakage_results = pd.DataFrame([
    {
        'model': 'safe feature vector',
        'feature_count_before_one_hot': len(safe_feature_columns),
        'roc_auc': round(roc_auc_score(y_test, safe_scores), 3),
        'average_precision': round(average_precision_score(y_test, safe_scores), 3),
        'precision_at_50': round(precision_at_k(y_test, safe_scores, 50), 3),
        'verdict': 'candidate feature set; still needs future-window validation later',
    },
    {
        'model': 'leaky feature vector',
        'feature_count_before_one_hot': len(leaky_feature_columns),
        'roc_auc': round(roc_auc_score(y_test, leaky_scores), 3),
        'average_precision': round(average_precision_score(y_test, leaky_scores), 3),
        'precision_at_50': round(precision_at_k(y_test, leaky_scores, 50), 3),
        'verdict': 'reject: includes target-derived trend fields',
    },
])

base_rate = round(y_test.mean(), 3)
print(f'Test-set base rate for is_declining_proxy: {base_rate}')
print(f'Train clients: {feature_vector.iloc[train_idx]["client_id"].nunique()} | Test clients: {feature_vector.iloc[test_idx]["client_id"].nunique()}')
print('\nLEAKAGE TEST RESULTS')
print(leakage_results.to_string(index=False))
leakage_results


Test-set base rate for is_declining_proxy: 0.517
Train clients: 24 | Test clients: 8

LEAKAGE TEST RESULTS
               model  feature_count_before_one_hot  roc_auc  average_precision  precision_at_50                                                           verdict
 safe feature vector                            38    0.608              0.592              0.5 candidate feature set; still needs future-window validation later
leaky feature vector                            46    1.000              1.000              1.0                      reject: includes target-derived trend fields


,model,feature_count_before_one_hot,roc_auc,average_precision,precision_at_50,verdict
0,safe feature vector,38,0.608,0.592,0.5,candidate feature set; still needs future-wind...
1,leaky feature vector,46,1.000,1.000,1.0,reject: includes target-derived trend fields


## 4. What I excluded and why

The excluded list is part of the model design, not an afterthought. I am refusing fields that either define the proxy, leak the proxy window, identify a row/group, or introduce context that is not part of the refresh decision.

This keeps the notebook aligned with the data contract: the model should support a human review queue, not memorize IDs or copy the target definition.


In [4]:
excluded = pd.DataFrame([
    ('trend_direction', 'defines is_declining_proxy directly; using it would give away the answer'),
    ('trend_pct', 'source calculation for trend_direction; label-derived leakage risk'),
    ('impressions_last_30d / impressions_prev_30d', 'direct inputs to the trend proxy; excluded for this proxy model'),
    ('clicks_last_30d / clicks_prev_30d', 'trend-window sibling fields; not needed for this starter proxy model'),
    ('sessions_last_30d / sessions_prev_30d', 'trend-window sibling fields; not needed for this starter proxy model'),
    ('content_id', 'pseudonymous row identifier; useful for traceability, not a model feature'),
    ('client_id', 'pseudonymous group identifier; use for grouped validation, not as a feature'),
    ('provider_used / model_used', 'generation metadata; not part of the reviewer action and may add bias/noise'),
    ('raw URLs / titles / queries / client names', 'not present in this public-safe dataset and should never be reconstructed'),
    ('product scores or decision flags', 'not present here; if rebuilt later, use only as a baseline, never as a discovery feature'),
], columns=['excluded_field_or_group', 'why_excluded'])

excluded


,excluded_field_or_group,why_excluded
0,trend_direction,defines is_declining_proxy directly; using it ...
1,trend_pct,source calculation for trend_direction; label-...
2,impressions_last_30d / impressions_prev_30d,direct inputs to the trend proxy; excluded for...
3,clicks_last_30d / clicks_prev_30d,trend-window sibling fields; not needed for th...
4,sessions_last_30d / sessions_prev_30d,trend-window sibling fields; not needed for th...
5,content_id,pseudonymous row identifier; useful for tracea...
6,client_id,pseudonymous group identifier; use for grouped...
7,provider_used / model_used,generation metadata; not part of the reviewer ...
8,raw URLs / titles / queries / client names,not present in this public-safe dataset and sh...
9,product scores or decision flags,"not present here; if rebuilt later, use only a..."


## Self-check

Before submitting, I checked each line honestly:

- [x] Every section above is filled with markdown thinking and code that backs it.
- [x] The notebook runs top to bottom with no errors.
- [x] No client names, URLs, or private queries are included.
- [x] My claims use careful words: observed, measured, proxy, directional, decision-support.
- [x] The final feature vector excludes label-derived trend fields and row/group IDs.
- [x] A deliberate leaky-feature test shows why the rejected fields are dangerous.
- [x] The work lives under `work/notebooks/`; after committing and pushing, I can submit my repo URL on the card.
